# Versionnage et traçage des données

Dans ce TP, vous versionnez des données avec [DVC](https://dvc.org/) et gardez la trace des traitements qui les produisent. DVC sépare le versionnage des données, dont git suit les métadonnées (des fichiers `.dvc`), de leur stockage, ici celui de [DagsHub](https://dagshub.com/), qui héberge aussi votre dépôt git.

Les commandes sont celles de la frame « Flux de travail de bout en bout » du cours. Une ligne qui commence par `!` est une commande du shell, où une variable Python s'insère entre accolades (`{token}`). Les étapes s'enchaînent : chacune part du dépôt de la précédente.

## Configuration & installations — À exécuter

Cette cellule installe DVC et son extension S3, puis vous place dans le dossier du projet, `~/dvc-project`. Réexécutez-la si l'environnement redémarre.

In [ ]:
!apt install tree
!pip install dvc dvc-s3
!mkdir -p ~/dvc-project
%cd ~/dvc-project

## 1. Créer le dépôt git

Créez un compte sur [DagsHub](https://dagshub.com/), puis un dépôt git vierge. Une fois le dépôt créé, cliquez sur « *Get started with Data* » et récupérez votre jeton secret (*token*) dans la partie « *Connection credentials* ».

Remplissez les variables `owner`, `repo` et `token` avec votre nom d'utilisateur DagsHub, le nom du dépôt et votre jeton secret, puis exécutez la cellule suivante : elle crée le dépôt git du projet, relié à votre dépôt DagsHub.

In [ ]:
owner = ""
repo = ""
token = ""

In [ ]:
!git init -b main
!git config user.name "Jeanne Durant"
!git config user.email "jeanne@durant.fr"
!git remote add origin https://{token}@dagshub.com/{owner}/{repo}.git

## 2. Initialiser DVC et son stockage distant

Initialisez DVC dans le dépôt, puis déclarez le stockage de votre dépôt DagsHub, compatible S3, comme stockage distant **par défaut** de DVC, avec l'adresse de son serveur et vos identifiants (`dvc remote modify`). DagsHub donne ces commandes dans l'onglet « *Data* » du bouton vert « *Remote* » de votre dépôt : ajoutez-y l'option qui fait de ce stockage celui par défaut. Commitez ensuite la configuration de DVC (le dossier `.dvc`) et poussez-la sur DagsHub.

Affichez `.dvc/config` et `.dvc/config.local` : lequel git suit-il, et pourquoi les identifiants sont-ils dans l'autre ?

**Documentation utile**

- https://dvc.org/doc/command-reference/remote/modify
- https://dvc.org/doc/user-guide/data-management/remote-storage/amazon-s3

In [ ]:
# Votre code ici

### Conseils supplémentaires

Nommez le stockage `origin`, d'adresse `s3://dvc` ; `-d` en fait le stockage par défaut, celui de `dvc push` et `dvc pull` sans `-r`. L'adresse du serveur, `endpointurl`, est `https://dagshub.com/{owner}/{repo}.s3`. Les deux identifiants, `access_key_id` et `secret_access_key`, valent tous deux votre jeton, et l'option `--local` les écrit dans `.dvc/config.local`, que le `.dvc/.gitignore` créé par `dvc init` exclut de git.

```python
!dvc init
!dvc remote add -d origin s3://dvc
!dvc remote modify origin endpointurl https://dagshub.com/{owner}/{repo}.s3
!dvc remote modify origin --local access_key_id {token}
!dvc remote modify origin ... secret_access_key {token}
!cat .dvc/config .dvc/config.local
!git add .dvc
!git commit -m "Initialise DVC"
!git push ...
```

### Solution

In [ ]:
!dvc init
!dvc remote add -d origin s3://dvc
!dvc remote modify origin endpointurl https://dagshub.com/{owner}/{repo}.s3
!dvc remote modify origin --local access_key_id {token}
!dvc remote modify origin --local secret_access_key {token}

In [ ]:
!cat .dvc/config .dvc/config.local

In [ ]:
!git add .dvc
!git commit -m "Initialise DVC"
!git push origin main

git suit `.dvc/config`, qui décrit le stockage pour toute l'équipe. `.dvc/config.local`, que `.dvc/.gitignore` exclut, garde les identifiants sur cette machine : ils ne partent jamais dans git, et chacun y met les siens (étape 8).

## 3. Récupérer une première version des données

Récupérez avec `dvc get` le fichier `wikipedia-movie-plots/wiki_movie_plots_deduped.csv` du dépôt `https://github.com/shuuchuu/datasets`, sous le nom `data.csv`, puis faites-le suivre par DVC avec `dvc add`.

Affichez ensuite `data.csv.dvc` (`!cat data.csv.dvc`) et le cache de DVC (`!tree .dvc/cache`) : que contient le fichier `.dvc`, et sous quel nom DVC range-t-il les données dans son cache ?

**Documentation utile**

- https://dvc.org/doc/command-reference/get

In [ ]:
# Votre code ici

### Conseils supplémentaires

`dvc get <dépôt> <chemin dans le dépôt> -o <fichier>` télécharge un fichier d'un dépôt git ou DVC, sans le suivre. `dvc add <fichier>` le copie dans le cache, écrit `<fichier>.dvc` et ajoute le fichier au `.gitignore`.

```python
!dvc get https://github.com/shuuchuu/datasets ... -o data.csv
!dvc add ...
!cat data.csv.dvc
!tree .dvc/cache
```

### Solution

In [ ]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc/cache

`data.csv.dvc` ne contient que le md5 du fichier, sa taille et son chemin : c'est ce petit fichier que git versionne. Le cache range les données sous leur md5 (`files/md5/<2 premiers caractères>/<le reste>`) : DVC les retrouve à partir du `.dvc`.

## 4. Enregistrer cette version

Commitez les fichiers que `dvc add` vous a dit d'ajouter à git, marquez ce commit de l'étiquette `v1`, et poussez la branche et l'étiquette sur DagsHub. Envoyez enfin les données sur le stockage distant avec `dvc push`.

Qu'envoie `git push`, et qu'envoie `dvc push` ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`dvc add` a affiché les fichiers à ajouter à git : `data.csv.dvc` et `.gitignore`. `git push origin main v1` pousse la branche et l'étiquette d'une seule commande, et `dvc push`, sans `-r`, envoie au stockage par défaut.

```python
!git add ... .gitignore
!git commit -m "Données brutes"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!git add data.csv.dvc .gitignore
!git commit -m "Données brutes"
!git tag v1

In [ ]:
!git push origin main v1
!dvc push

`git push` n'envoie que les métadonnées : `data.csv.dvc` et `.gitignore`, quelques lignes. `dvc push` envoie le fichier de données lui-même (81 Mo), depuis le cache, vers le stockage.

## 5. Modifier les données

La cellule suivante modifie les données : elle trie les lignes de `data.csv` dans l'ordre inverse. Exécutez-la, puis enregistrez cette nouvelle version comme la première, avec l'étiquette `v2`, et envoyez-la sur DagsHub (git et DVC).

Comparez `data.csv.dvc` et le cache de DVC à ce qu'ils étaient à l'étape 3 : qu'est-ce qui a changé ?

In [ ]:
!sort -r data.csv -o data.csv

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les commandes des étapes 3 et 4 : `dvc add` recalcule le md5 et met la nouvelle version dans le cache, `git add` enregistre le nouveau `data.csv.dvc` (le `.gitignore` n'a pas changé).

```python
!dvc add data.csv
!cat data.csv.dvc
!tree .dvc/cache
!git add ...
!git commit -m "Données v2"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc/cache

In [ ]:
!git add data.csv.dvc
!git commit -m "Données v2"
!git tag v2

In [ ]:
!git push origin main v2
!dvc push

Le md5 de `data.csv.dvc` a changé, pas la taille : les mêmes lignes, dans un autre ordre. Le cache contient maintenant les deux versions, chacune sous son md5 : revenir à la première ne demande pas de la télécharger à nouveau.

## 6. Revenir aux données originales

Vous découvrez un problème dans les nouvelles données : revenez à la première version. Restaurez avec `git restore` le `data.csv.dvc` de l'étiquette `v1`, puis, avec `dvc checkout`, le fichier de données qu'il désigne. Commitez ce retour en arrière, avec l'étiquette `v3`, et envoyez-le sur DagsHub.

Pourquoi `dvc push` n'a-t-il rien à envoyer cette fois ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`git restore --source v1 <fichier>` remet le fichier tel qu'il était à `v1`, sans changer de branche. `dvc checkout` remet alors dans le dossier de travail, depuis le cache, les données que désignent les fichiers `.dvc`.

```python
!git restore --source ... data.csv.dvc
!dvc checkout
!cat data.csv.dvc
!git add data.csv.dvc
!git commit -m "Retour aux données v1"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!git restore --source v1 data.csv.dvc
!dvc checkout

In [ ]:
!cat data.csv.dvc

In [ ]:
!git add data.csv.dvc
!git commit -m "Retour aux données v1"
!git tag v3

In [ ]:
!git push origin main v3
!dvc push

Les données de `v1` sont déjà sur le stockage, sous le même md5 : DVC n'envoie que ce que le stockage n'a pas. L'historique git garde les trois versions, et `v2` reste récupérable de la même façon.

## 7. Une étape de pipeline

La cellule suivante écrit le script `upper.py`, qui recopie le fichier donné en premier argument dans celui donné en second, en majuscules. Exécutez-la.

Ajoutez avec `dvc stage add` une étape `uppercase` qui produit `data-upper.csv` à partir de `data.csv` avec ce script, en déclarant le script et les données comme dépendances. Exécutez-la avec `dvc repro`, puis enregistrez dans git le script et la pipeline, avec le fichier `dvc.lock` que `dvc repro` écrit, et envoyez sa sortie sur le stockage.

Relancez ensuite `dvc repro` : que fait-il, et pourquoi ? Que contient `dvc.lock` ?

In [ ]:
%%writefile upper.py
from pathlib import Path
from sys import argv

text = Path(argv[1]).read_text(encoding="utf-8")
Path(argv[2]).write_text(text.upper(), encoding="utf-8")

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les options de la frame « Définir une étape » du cours : `--name` (le nom de l'étape), `--deps` (une fois par dépendance), `--outs` (la sortie), puis la commande. `dvc stage add` écrit `dvc.yaml` et ajoute la sortie au `.gitignore` ; `dvc repro` exécute l'étape et écrit `dvc.lock`.

```python
!dvc stage add --name uppercase --deps ... --deps ... --outs ... python upper.py data.csv data-upper.csv
!dvc repro
!git add upper.py dvc.yaml ... .gitignore
!git commit -m "Pipeline en majuscules"
!git push origin main
!dvc push
!dvc repro
!cat dvc.lock
```

### Solution

In [ ]:
!dvc stage add --name uppercase --deps upper.py --deps data.csv --outs data-upper.csv python upper.py data.csv data-upper.csv

In [ ]:
!dvc repro

In [ ]:
!git add upper.py dvc.yaml dvc.lock .gitignore
!git commit -m "Pipeline en majuscules"
!git push origin main
!dvc push

In [ ]:
!dvc repro
!cat dvc.lock

Le second `dvc repro` saute l'étape : il compare les md5 de ses dépendances et de sa sortie à ceux de `dvc.lock`, et rien n'a changé. `dvc.lock` note la commande et ces md5 : il dit exactement à partir de quoi et comment `data-upper.csv` a été produit, c'est le traçage des données.

## 8. Sur une autre machine

Une collègue rejoint le projet. Clonez votre dépôt DagsHub dans `~/other-machine`, comme elle le ferait sur sa machine, et placez-vous dans ce dossier avec la commande magique `%cd`. Les identifiants du stockage sont restés dans votre `.dvc/config.local` : donnez-les au clone, puis récupérez les données avec `dvc pull`.

Le clone a-t-il `data.csv` et `data-upper.csv` ? Que disent `dvc status` et `dvc repro` ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`git clone <adresse> ~/other-machine`, avec l'adresse du dépôt de l'étape 1 ; les deux `dvc remote modify ... --local` de l'étape 2 ; `dvc pull` récupère toutes les données que désignent les fichiers `.dvc` et `dvc.lock` du commit courant.

```python
!git clone https://{token}@dagshub.com/{owner}/{repo}.git ...
%cd ~/other-machine
!dvc remote modify origin --local ... {token}
!dvc remote modify origin --local ... {token}
!dvc pull
!ls
!dvc status
!dvc repro
```

### Solution

In [ ]:
!git clone https://{token}@dagshub.com/{owner}/{repo}.git ~/other-machine

In [ ]:
%cd ~/other-machine

In [ ]:
!dvc remote modify origin --local access_key_id {token}
!dvc remote modify origin --local secret_access_key {token}

In [ ]:
!dvc pull

In [ ]:
!ls
!dvc status
!dvc repro

`dvc pull` a récupéré `data.csv`, dans sa version de `v3`, et `data-upper.csv`, sans réexécuter le script. `dvc status` dit que les données et la pipeline sont à jour, et `dvc repro` n'a rien à refaire : le clone a exactement les données et les traitements du projet.